In [ ]:
!pip install pyAudioAnalysis
!pip install eyed3
!pip install pydub

In [6]:
import numpy as np
import os
from pydub import AudioSegment
from pyAudioAnalysis import audioBasicIO, ShortTermFeatures

In [8]:
video_path = "/content/Video Interviewing Example.mp4"
audio_path = "/content/Video Interviewing Example.wav"

if not os.path.exists(audio_path):
    print(f"Converting {video_path} to {audio_path}...")
    audio = AudioSegment.from_file(video_path, format="mp4")
    audio.export(audio_path, format="wav")
    print("Conversion complete.")
else:
    print(f"Using existing audio file: {audio_path}")

# read the WAV file with pyAudioAnalysis
Fs, x = audioBasicIO.read_audio_file(audio_path)
x = audioBasicIO.stereo_to_mono(x)

Using existing audio file: /content/Video Interviewing Example.wav


In [10]:
# window = 50 ms, step = 25 ms
win = int(0.050 * Fs)
step = int(0.025 * Fs)
F, f_names = ShortTermFeatures.feature_extraction(x, Fs, win, step)

try:
    energy_idx = f_names.index('energy')
except ValueError:
    print("Available feature names:", f_names)
    energy_idx = 1 # Fallback in case 'energy' is not found

energy_frames = F[energy_idx, :].astype(float)
energy_db = 10.0 * np.log10(np.maximum(energy_frames, 1e-12))





# --- Loudness calculations ---
mean_energy_db = float(np.mean(energy_db))
median_energy_db = float(np.median(energy_db))
std_energy_db = float(np.std(energy_db))

def _map_db_to_0_100(mean_db, ref_db=median_energy_db, min_offset=-35.0, max_offset=20.0):
    min_db = ref_db + min_offset
    max_db = ref_db + max_offset
    score = (mean_db - min_db) / (max_db - min_db) * 100.0
    return float(max(0.0, min(100.0, score)))

avg_loudness = _map_db_to_0_100(mean_energy_db)





# --- Expressiveness calculations ---
variation = float(max(0.0, min(100.0, (std_energy_db / 8.0) * 100.0)))



# --- Confidence (Speech) calculation ---
# Derived from a steady vocal tone and controlled loudness, reflecting a comfortable presence.

# Loudness factor: Higher if balanced, lower if too calm or too loud.
# Ideal range for balanced loudness is approximately 30-60 (based on prior analysis).
if 30 <= avg_loudness < 60:
    loudness_factor = 1.0
elif avg_loudness < 30: # Too calm
    # Scale down from 0.0 to 0.7 as avg_loudness approaches 30
    loudness_factor = max(0.0, avg_loudness / 30 * 0.7)
else: # Too loud
    # Scale down from 0.7 to 0.0 as avg_loudness goes from 60 to 100
    loudness_factor = max(0.0, (100 - avg_loudness) / 40 * 0.7)

# Expressiveness factor: Higher if flat/monotone or moderately expressive, lower if highly expressive.
if variation < 10: # Flat / Monotone (most steady)
    expressiveness_factor = 1.0
elif variation < 40: # Moderately Expressive
    expressiveness_factor = 0.8
else: # Highly Expressive (least steady for "controlled tone")
    expressiveness_factor = 0.5

# Combine factors for a final confidence score (equal weighting given to loudness and expressiveness)
confidence_score = (loudness_factor * 0.5 + expressiveness_factor * 0.5) * 100

In [11]:
print(f"\n=== Speech Confidence Analysis ===")
print(f"  Loudness Score Component: {loudness_factor*100:.1f}/100")
print(f"  Expressiveness Score Component: {expressiveness_factor*100:.1f}/100")
print(f"\n  Final Speech Confidence Score: {confidence_score:.1f}/100")

# Interpretation
if confidence_score >= 80:
    confidence_label = "Excellent – highly confident and composed delivery."
elif confidence_score >= 60:
    confidence_label = "Good – generally confident with minor fluctuations."
elif confidence_score >= 40:
    confidence_label = "Moderate – some signs of hesitation or lack of control."
else:
    confidence_label = "Developing – noticeable inconsistencies in tone and loudness."

print(f"  Interpretation: {confidence_label}")


=== Speech Confidence Analysis ===
  Loudness Score Component: 100.0/100
  Expressiveness Score Component: 50.0/100

  Final Speech Confidence Score: 75.0/100
  Interpretation: Good – generally confident with minor fluctuations.
